In [1]:
import numpy as np

tab = np.random.rand(100, 3, 3)

In [2]:
def mult_end():
    tableau = np.zeros([100, 3, 3])
    for index, tab in enumerate(tableau):
        tab += index
        
    tab = tab * 2 
    return tableau 

def mult_in():
    tableau = np.zeros([100, 3, 3])
    for index, tab in enumerate(tableau):
        tab += 2 * index
        
    return tableau 

In [3]:
%timeit mult_end()

235 µs ± 20.7 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [4]:
%timeit mult_in()

230 µs ± 9.27 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [5]:
def compute_angle_alignment_long_axis(polarity_angles, ellipse_long_axis_angles, interaction_strength, angular_noise, delta_t):
    """
    #TODO
    d_theta = alpha * sin(theta - ellipse_long_axis_angles) + angular_noise
    """
    
    # Version numpy 
    # Should take care of 'inf' values in ellipse_long_axis_angles which is why an individual computation may be better ?
    # Or process immediately the angle 
    
    angles_next = delta_t * (interaction_strength * np.sin(polarity_angles - ellipse_long_axis_angles) + angular_noise) + polarity_angles
    
    return angles_next
    # Should take care of 'inf' values in ellipse_long_axis_angles which is why an individual computation may be better ?
    # Or process immediately the angle 
    

In [6]:
nb_cells = 10
np.random.seed(42)
polarity_angles = np.random.rand(nb_cells) * 2*np.pi
ellipse_long_axis_angles = np.random.rand(nb_cells) * 2*np.pi
interaction_strength = np.ones(nb_cells)
angular_noise = np.ones(nb_cells) * 0.0
delta_t = 0.001

In [7]:
angles_next = compute_angle_alignment_long_axis(polarity_angles, ellipse_long_axis_angles, interaction_strength, angular_noise, delta_t)

In [8]:
angles_next

array([2.35409913, 5.97339384, 4.59866352, 3.76213726, 0.98013259,
       0.97997111, 0.36395039, 5.44318475, 3.77779079, 4.44945026])

In [9]:
polarity_angles

array([2.35330497, 5.97351416, 4.59925358, 3.76148219, 0.98029403,
       0.98014248, 0.3649501 , 5.44234523, 3.77691701, 4.44895122])

In [10]:
def gaussian_white_noise_process(mean, std, sample_number):
    """
    Draws random samples from a gaussian distribution

    Parameters:
    mean : double
        Mean of the Gaussian Distribution
    std : double 
        Standard deviation of the white noise process
    sample_number : int
        Number of samples to generate
    ----- 
    Returns:
    white_noise_samples : numpy array
        Array of random white noise samples (double)
    """

    white_noise_samples = np.random.normal(mean, std, sample_number)
    return white_noise_samples


### Rewrite these functions
def compute_random_rotational_diffusion_euler(angle, random_process, delta_t):
    """
    Compute the next value of an angle for the polarity vector using Giavazzi (3) and Euler's method

    Parameters:
    angle : double 
        Theta value in radian for the polarity vector.
    random_process : double
        Random value of a white noise process
    delta_t : double
        Time integration step
    -----    
    Returns:
    angle_next : numpy array
        Theta value in radian for the next timestep
    """

    angle_next = delta_t * random_process + angle 

    # Need to map over [0;2pi[
    # while(angle_next >= 2*np.pi):
    #     angle_next = angle_next - 2*np.pi
    # while(angle_next < 0):
    #     angle_next = angle_next + 2*np.pi


    # Modulo 2 pi to normalize
    angle_next = angle_next % (2*np.pi)





    return angle_next



def compute_every_random_rotational_diffusion_euler(angles, angular_noise, delta_t):
    """
    Compute the values at next timestep for every angle assuming random rotational diffusion (Giavazzi (3)) 
    and using Euler's method

    Parameters:
    angles: numpy array
        Theta values in radian for the polarity vector
    angular_noise : double
        D_r term, determines the memory of stochastic noise in the system
    delta_t : double
        Time integration step
    -----    
    Returns:
    angles_next : numpy array
        Theta values in radian for the next timestep
    """

    # Generate an array of samples from a white noise process of zero mean and var = 2*D_r
    random_process_samples = gaussian_white_noise_process(0, angular_noise*2, len(angles)) 

    angles_next = []

    for angle, random_sample in zip(angles, random_process_samples):
        angle_next = compute_random_rotational_diffusion_euler(angle, random_sample, delta_t) 
        # Store the newly computed angle in a list
        angles_next.append(angle_next)

    # Convert to numpy array
    angles_next = np.array(angles_next)
    return angles_next 



In [11]:
def compute_random_diff(angles, angular_noise,delta_t):
    """
    # TODO
    """
    
    random_process_samples = gaussian_white_noise_process(0, angular_noise*2, len(angles)) 
    angles_next = delta_t * random_process_samples + angles 

    # Modulo 2 pi to normalize
    angles_next = angles_next % (2*np.pi)

    return angles_next 


In [12]:
def compute_random_diff_2(angles, angular_noise,delta_t):
    """
    # TODO
    """
    
    random_process_samples = gaussian_white_noise_process(0, angular_noise*2, len(angles)) 
    angles_next = delta_t * random_process_samples + angles 

    # Modulo 2 pi to normalize
    angles_next = np.mod(angles_next,2*np.pi)

    return angles_next 

In [13]:
np.random.seed(2019)
angles = np.random.rand(10000) * 2*np.pi
angular_noise = 0.1
delta_t = 0.001

a1 = compute_random_diff(angles, angular_noise, delta_t)
a2 = compute_every_random_rotational_diffusion_euler(angles, angular_noise, delta_t)
a3 = compute_random_diff_2(angles, angular_noise, delta_t)

In [14]:
%timeit compute_random_diff(angles, angular_noise, delta_t)

720 µs ± 45.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [15]:
%timeit compute_every_random_rotational_diffusion_euler(angles, angular_noise, delta_t) 

8.61 ms ± 222 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [16]:
%timeit compute_random_diff_2(angles, angular_noise, delta_t)

685 µs ± 8.61 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [17]:
print(a1)
print('-'*10)
print(a3)

[5.67678713 2.46991518 3.92059397 ... 3.34687622 3.28408785 5.59010751]
----------
[5.67656164 2.4695493  3.92067775 ... 3.34661745 3.28432091 5.58980991]


In [18]:
def generate_every_polarity_vector(angles):
    """
    Generate the polarity vectors [cos(angle),sin(angle)] components for every point

    Parameter:
    angles : numpy array
     Polarity angles in radian
    -----
    Returns:
    polarities_vector : numpy array of numpy array
     Array of [cos(angle), sin(angle)] components for every point
    """

    polarities_vector = []
    # Call the np.cos directly over the whole angles array (timed it, it's nicer than the previous solution)
    polarities_x = np.cos(angles)
    polarities_y = np.sin(angles)

    for polarity_x, polarity_y in zip(polarities_x, polarities_y):
        polarities_vector.append([polarity_x, polarity_y])

    # Convert into a numpy array
    polarities_vector = np.array(polarities_vector)

    return polarities_vector

In [19]:
def generate_every_polarity_vector_np(angles):
    """
    Generate the polarity vectors [cos(angle),sin(angle)] components for every point

    Parameter:
    angles : numpy array
     Polarity angles in radian
    -----
    Returns:
    polarities_vector : numpy array of numpy array
     Array of [cos(angle), sin(angle)] components for every point
    """

    polarities_vector = np.zeros([len(angles),2]) 
    # Call the np.cos directly over the whole angles array (timed it, it's nicer than the previous solution)
    polarities_x = np.cos(angles)
    polarities_y = np.sin(angles)

    polarities_vector[:,0] = polarities_x[:]
    polarities_vector[:,1] = polarities_y[:]
    return polarities_vector

In [20]:
generate_every_polarity_vector(angles) == generate_every_polarity_vector_np(angles)

array([[ True,  True],
       [ True,  True],
       [ True,  True],
       ...,
       [ True,  True],
       [ True,  True],
       [ True,  True]])

In [21]:
%timeit generate_every_polarity_vector(angles)

6.33 ms ± 330 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [22]:
%timeit generate_every_polarity_vector_np(angles)

90.2 µs ± 2.58 µs per loop (mean ± std. dev. of 7 runs, 10000 loops each)


In [23]:
def compute_every_force_on_cell_euler(points,regions,vertices, delta_r, K_A, K_P, A_0, P_0):
    """
    Compute the total force exerted on a cell F = -dE/dr using "Euler's method"
    Seed center points are successively moved in the forward and backward x and y directions by a quantity 
    delta_r, and the modified tissue energy is computed for each of these movements
    dE/dr -> (E(x+delta,y)-E(x,y))/delta + (E(x,y+delta)-E(x,y))/delta
    WARNING: Putting a delta too high may raise a KeyError in conjunction with the voronoi_finite_polygons_2d method
    
    Parameters:
    ------
    points : 2D array of (double)
        The seed points (centers) of Voronoi tesselation.
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    delta_r : float
        Step of integration taken equal for delta_x and delta_y
    K_A : float
        Area "stiffness"
    K_P : float
        Perimeter "stiffness"
    A_0 : float
        Target value for area
    P_0 : float
        Target value for perimeter


    Returns:
    ------
    force : 2D ndarray
        The i-th row contains the x and y component for the force applied to the i-th cell with the i-th center (seed)
    """

 


    # Compute the tissue energy prior any displacement :
    indx, perimeters, areas = get_polygons_properties_2d(regions,vertices)
    energy_old = compute_every_cell_energy(perimeters, areas, K_A, K_P, A_0, P_0) 

    # diff_energy_x = E(x+delta_x,y) - E(x,y)
    # diff_energy_y = E(x,y+delta_y) - E(x,y)
    # The arrays indices correspond to the POINTS' indices
    array_diff_energy_x = np.zeros(len(points))
    array_diff_energy_y = np.zeros(len(points))

    pnt_number = len(points)

    for index, point in enumerate(points):

        ## X DISPLACEMENT FORWARD
        # Save the value of the point about to be displaced
        point_save_x = point[0]
        # Displace the x coordinate of a point forward by delta_r_x (delta_r) 
        points[index][0] = point[0] + delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor)
        # regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        # indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x, vertices_modified_x)
        indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x[0:pnt_number], vertices_modified_x)

        # Compute the new tissue energy (after a delta_r_x displacement of point[index]) and save it
        energy_modified_x_forward = compute_every_cell_energy(perimeters_modified_x, areas_modified_x, K_A, K_P, A_0, P_0)

        # Replace the displaced point at its original position
        # FIXME : Check if the point_save is distinct from points[index][0]
        points[index][0] = point_save_x

        ## X DISPLACEMENT BACKWARD
        # Save the value of the point about to be displaced
        point_save_x = point[0]
        # Displace the x coordinate of a point backward by delta_r_x (delta_r) 
        points[index][0] = point[0] - delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor)
        # regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        # indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x, vertices_modified_x)
        indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x[0:pnt_number], vertices_modified_x)

        # Compute the new tissue energy (after a delta_r_x displacement of point[index]) and save it
        energy_modified_x_backward = compute_every_cell_energy(perimeters_modified_x, areas_modified_x, K_A, K_P, A_0, P_0)

        # Replace the displaced point at its original position
        points[index][0] = point_save_x


        ## X DERIVATIVE COMPUTATION
        array_diff_energy_x[index] = energy_modified_x_forward - energy_modified_x_backward
        # array_diff_energy_x[index] = energy_modified_x_forward - energy_old



        ############################# 

        ## Y DISPLACEMENT FORWARD
        # Do the same thing for the y coordinate

        # Save the value of the point about to be displaced
        point_save_y = point[1]
        # Displace the y coordinate of a point forward by delta_r_y (delta_r) 
        points[index][1] = point[1] + delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor)
        # regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        indices_modified_y, perimeters_modified_y, areas_modified_y = get_polygons_properties_2d(regions_modified_y[0:pnt_number], vertices_modified_y)

        # Compute the new tissue energy (after a delta_r_y displacement of point[index]) and save it
        energy_modified_y_forward = compute_every_cell_energy(perimeters_modified_y, areas_modified_y, K_A, K_P, A_0, P_0)
        # Replace the displaced point at its original position
        points[index][1] = point_save_y

        ## Y DISPLACEMENT BACKWARD
        # Do the same thing for the y coordinate

        # Save the value of the point about to be displaced
        point_save_y = point[1]
        # Displace the y coordinate of a point backward by delta_r_y (delta_r) 
        points[index][1] = point[1] - delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor)
        # regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        indices_modified_y, perimeters_modified_y, areas_modified_y = get_polygons_properties_2d(regions_modified_y[0:pnt_number], vertices_modified_y)

        # Compute the new tissue energy (after a delta_r_y displacement of point[index]) and save it
        energy_modified_y_backward = compute_every_cell_energy(perimeters_modified_y, areas_modified_y, K_A, K_P, A_0, P_0)
        # Replace the displaced point at its original position
        points[index][1] = point_save_y

        ## Y DERIVATIVE COMPUTATION
        array_diff_energy_y[index] = energy_modified_y_forward - energy_modified_y_backward


        ### Debug: uncomment if need to register the modifications to the areas and perimeter
        ### during the spatial derivative part.
        # Log the following parameters in the appropriate files
        # One written row : $(point_numbers) areas/perimeters after the displacement of one seedpoint by delta_r_r
        # create_csv_log_file("areas_log_modified_x.csv", areas_modified_x)
        # create_csv_log_file("areas_log_modified_y.csv", areas_modified_y )
        # create_csv_log_file("perimeters_log_modified_x.csv", perimeters_modified_x)
        # create_csv_log_file("perimeters_log_modified_y.csv", perimeters_modified_y)



    # ON CHERCHE LES ENERGIES DERIVEES PAR RAPPORT AU CENTRE R_I
    dE_drix = array_diff_energy_x/(2*delta_r)
    dE_driy = array_diff_energy_y/(2*delta_r)
    # dE_drix = array_diff_energy_x/delta_r
    # dE_driy = array_diff_energy_y/delta_r

    # Write into a log file
    create_dE_drixy_csv_log_file(dE_drix,dE_driy)

    # 2D array to store the x and y components
    # force = np.array([-dE_drix, -dE_driy])
    ## Instead let us store like [ [f_0x, f_0y], [f_1x,f_1y]... [f_nx,f_ny]]
    # Naively
    forces = []
    for x_component, y_component in zip(dE_drix, dE_driy):
        forces.append([x_component, y_component])
    
    # Convert into a numpy array
    forces = np.array(forces)
    
    # DEBUG 
    forces_norm = np.linalg.norm(forces, axis=1)
    # Append into the force logger
    force_logger.append(forces_norm)

    return forces


In [24]:
def compute_every_force_on_cell_euler_np(points,regions,vertices, delta_r, K_A, K_P, A_0, P_0):
    """
    Compute the total force exerted on a cell F = -dE/dr using "Euler's method"
    Seed center points are successively moved in the forward and backward x and y directions by a quantity 
    delta_r, and the modified tissue energy is computed for each of these movements
    dE/dr -> (E(x+delta,y)-E(x,y))/delta + (E(x,y+delta)-E(x,y))/delta
    WARNING: Putting a delta too high may raise a KeyError in conjunction with the voronoi_finite_polygons_2d method
    
    Parameters:
    ------
    points : 2D array of (double)
        The seed points (centers) of Voronoi tesselation.
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    delta_r : float
        Step of integration taken equal for delta_x and delta_y
    K_A : float
        Area "stiffness"
    K_P : float
        Perimeter "stiffness"
    A_0 : float
        Target value for area
    P_0 : float
        Target value for perimeter


    Returns:
    ------
    force : 2D ndarray
        The i-th row contains the x and y component for the force applied to the i-th cell with the i-th center (seed)
    """

 


    # Compute the tissue energy prior any displacement :
    indx, perimeters, areas = get_polygons_properties_2d(regions,vertices)
    energy_old = compute_every_cell_energy(perimeters, areas, K_A, K_P, A_0, P_0) 

    # diff_energy_x = E(x+delta_x,y) - E(x,y)
    # diff_energy_y = E(x,y+delta_y) - E(x,y)
    # The arrays indices correspond to the POINTS' indices
    array_diff_energy_x = np.zeros(len(points))
    array_diff_energy_y = np.zeros(len(points))
    forces = np.zeros(len(points))
    pnt_number = len(points)

    for index, point in enumerate(points):

        ## X DISPLACEMENT FORWARD
        # Save the value of the point about to be displaced
        point_save_x = point[0]
        # Displace the x coordinate of a point forward by delta_r_x (delta_r) 
        points[index][0] = point[0] + delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor)
        # regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        # indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x, vertices_modified_x)
        indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x[0:pnt_number], vertices_modified_x)

        # Compute the new tissue energy (after a delta_r_x displacement of point[index]) and save it
        energy_modified_x_forward = compute_every_cell_energy(perimeters_modified_x, areas_modified_x, K_A, K_P, A_0, P_0)

        # Replace the displaced point at its original position
        # FIXME : Check if the point_save is distinct from points[index][0]
        points[index][0] = point_save_x

        ## X DISPLACEMENT BACKWARD
        # Save the value of the point about to be displaced
        point_save_x = point[0]
        # Displace the x coordinate of a point backward by delta_r_x (delta_r) 
        points[index][0] = point[0] - delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor)
        # regions_modified_x, vertices_modified_x = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        # indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x, vertices_modified_x)
        indices_modified_x, perimeters_modified_x, areas_modified_x = get_polygons_properties_2d(regions_modified_x[0:pnt_number], vertices_modified_x)

        # Compute the new tissue energy (after a delta_r_x displacement of point[index]) and save it
        energy_modified_x_backward = compute_every_cell_energy(perimeters_modified_x, areas_modified_x, K_A, K_P, A_0, P_0)

        # Replace the displaced point at its original position
        points[index][0] = point_save_x


        ## X DERIVATIVE COMPUTATION
        array_diff_energy_x[index] = energy_modified_x_forward - energy_modified_x_backward
        # array_diff_energy_x[index] = energy_modified_x_forward - energy_old



        ############################# 

        ## Y DISPLACEMENT FORWARD
        # Do the same thing for the y coordinate

        # Save the value of the point about to be displaced
        point_save_y = point[1]
        # Displace the y coordinate of a point forward by delta_r_y (delta_r) 
        points[index][1] = point[1] + delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor)
        # regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        indices_modified_y, perimeters_modified_y, areas_modified_y = get_polygons_properties_2d(regions_modified_y[0:pnt_number], vertices_modified_y)

        # Compute the new tissue energy (after a delta_r_y displacement of point[index]) and save it
        energy_modified_y_forward = compute_every_cell_energy(perimeters_modified_y, areas_modified_y, K_A, K_P, A_0, P_0)
        # Replace the displaced point at its original position
        points[index][1] = point_save_y

        ## Y DISPLACEMENT BACKWARD
        # Do the same thing for the y coordinate

        # Save the value of the point about to be displaced
        point_save_y = point[1]
        # Displace the y coordinate of a point backward by delta_r_y (delta_r) 
        points[index][1] = point[1] - delta_r

        # DEBUG : Add periodic boundary generation HERE
        extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

        # Recompute the tesselation
        vor = Voronoi(extended_points) 
        # Reconstruct finite regions 
        regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor)
        # regions_modified_y, vertices_modified_y = voronoi_finite_polygons_2d(vor, boxsize*2.0)
        indices_modified_y, perimeters_modified_y, areas_modified_y = get_polygons_properties_2d(regions_modified_y[0:pnt_number], vertices_modified_y)

        # Compute the new tissue energy (after a delta_r_y displacement of point[index]) and save it
        energy_modified_y_backward = compute_every_cell_energy(perimeters_modified_y, areas_modified_y, K_A, K_P, A_0, P_0)
        # Replace the displaced point at its original position
        points[index][1] = point_save_y

        ## Y DERIVATIVE COMPUTATION
        array_diff_energy_y[index] = energy_modified_y_forward - energy_modified_y_backward


        ### Debug: uncomment if need to register the modifications to the areas and perimeter
        ### during the spatial derivative part.
        # Log the following parameters in the appropriate files
        # One written row : $(point_numbers) areas/perimeters after the displacement of one seedpoint by delta_r_r
        # create_csv_log_file("areas_log_modified_x.csv", areas_modified_x)
        # create_csv_log_file("areas_log_modified_y.csv", areas_modified_y )
        # create_csv_log_file("perimeters_log_modified_x.csv", perimeters_modified_x)
        # create_csv_log_file("perimeters_log_modified_y.csv", perimeters_modified_y)



    # ON CHERCHE LES ENERGIES DERIVEES PAR RAPPORT AU CENTRE R_I
    dE_drix = array_diff_energy_x/(2*delta_r)
    dE_driy = array_diff_energy_y/(2*delta_r)
    # dE_drix = array_diff_energy_x/delta_r
    # dE_driy = array_diff_energy_y/delta_r

    # Write into a log file
    create_dE_drixy_csv_log_file(dE_drix,dE_driy)

    # 2D array to store the x and y components
    # force = np.array([-dE_drix, -dE_driy])
    ## Instead let us store like [ [f_0x, f_0y], [f_1x,f_1y]... [f_nx,f_ny]]
    
    forces[:,0] = dE_drix[:]
    forces[:,1] = dE_driy[:]
    
    # DEBUG 
    forces_norm = np.linalg.norm(forces, axis=1)
    # Append into the force logger
    force_logger.append(forces_norm)

    return forces


In [25]:
%load_ext cython

In [26]:
##### IMPORTS FROM VORONOI TESSELATION
def voronoi_finite_polygons_2d(vor, radius=None):
    """
    Reconstruct infinite voronoi regions in a 2D diagram to finite regions.

    Parameters
    ----------
    vor : Voronoi
        Input diagram
    radius : float, optional
        Distance to 'points at infinity'.

    Returns
    -------
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
        Vertices are sorted counter-clockwise.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.

    """

    if vor.points.shape[1] != 2:
        raise ValueError("Requires 2D input")

    new_regions = []
    new_vertices = vor.vertices.tolist()

    center = vor.points.mean(axis=0)
    if radius is None:
        radius = vor.points.ptp().max()*2

    # Construct a map containing all ridges for a given point
    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    # Reconstruct infinite regions
    for p1, region in enumerate(vor.point_region):
        vertices = vor.regions[region]

        if all(v >= 0 for v in vertices):
            # finite region
            new_regions.append(vertices)
            continue

        # reconstruct a non-finite region
        ridges = all_ridges[p1]
        new_region = [v for v in vertices if v >= 0]

        for p2, v1, v2 in ridges:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                # finite ridge: already in the region
                continue

            # Compute the missing endpoint of an infinite ridge

            t = vor.points[p2] - vor.points[p1] # tangent
            t /= np.linalg.norm(t)
            n = np.array([-t[1], t[0]])  # normal

            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, n)) * n
            far_point = vor.vertices[v2] + direction * radius

            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())


        # sort region counterclockwise
        vs = np.asarray([new_vertices[v] for v in new_region])
        c = vs.mean(axis=0)
        angles = np.arctan2(vs[:,1] - c[1], vs[:,0] - c[0])
        new_region = np.array(new_region)[np.argsort(angles)]

        # finish
        new_regions.append(new_region.tolist())

    return new_regions, np.asarray(new_vertices)



def get_polygons_properties_2d(regions, vertices):
    """
    Extract the perimeter and area for each finite polygon in 2D
    Partially Cython typed.
    
    Parameters
    -----
    regions : numpy array
        Indices of vertices in each revised Voronoi regions.
    vertices : numpy array
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end.
    
    Returns
    -----
    properties_index : numpy array
        List of index for each region in regions treated (probably not useful since perimeter and area are
        already acquired in the order of region in regions)
    properties_perimeter : numpy array
        Perimeter of each region in regions treated
    properties_area : numpy array
        Area of each region in regions treated
        Warning : In 2D : ConvexHull.area is the perimeter, while ConvexHull.volume is the area
    """
    
    ## Typing variables
    len_regions = len(regions)
    
    properties_index_np = np.zeros(len_regions)
    properties_perimeter_np = np.zeros(len_regions)
    properties_area_np = np.zeros(len_regions)
    
    properties_index = properties_index_np
    properties_perimeter = properties_perimeter_np
    properties_area = properties_area_np

    for index in range(len_regions):
        polygon = vertices[regions[index]]
        properties_index[index] = index
        convHull = ConvexHull(polygon)
        convHull_area = convHull.area
        convHull_volume = convHull.volume
        properties_perimeter[index] = convHull_area
        properties_area[index] = convHull_volume
        
    return properties_index,properties_perimeter,properties_area


def plot_polygons(vor, regions, vertices):
    """
    Graphical function to plot colored polygons within a box

    Note that while the Voronoi regions are finite, they may still go beyond the box 
    Parameters:
    vor : A Voronoi object
    regions : list of lists
        Indices of vertices in each revised Voronoi regions.
    vertices : list of lists
        Coordinates for revised Voronoi vertices. Same as coordinates
        of input vertices, with 'points at infinity' appended to the
        end. 
    """
    for region in regions:
        polygon = vertices[region]
        # plt.fill(*zip(*polygon), alpha=0.4) # Color the polygons
        plt.plot(polygon[:,0], polygon[:,1], '--k')
        ## Uncomment the lines below for a nicer display in an area of interest
        # plt.xlim(vor.min_bound[0] - 0.1, vor.max_bound[0] + 0.1) # Forces display in a box
        # plt.ylim(vor.min_bound[1] - 0.1, vor.max_bound[1] + 0.1) # Forces display in a box


def initialization(point_number, boxsize):
    """
    Create an array of random seed points (Voronoi center), and applies periodic boundary conditions.
    TO DO: initialize polarity, angles, momentum, a flag indicating at which stage we are currently in.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float 
        Length of the axis for observation box (assumed square for now)

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of random seed points (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # XXX : DEBUG
    generated_points = np.random.rand(point_number, 2) * boxsize - (boxsize * 0.5 * 0.5)
    # XXX : With a Poisson distribution
    # generated_points = np.random.poisson(lam=1.0,size=(point_number, 2)) * boxsize - (boxsize * 0.5 * 0.5)
    # generated_points = (2* np.random.rand(point_number, 2) - 1) * boxsize * 0.5
    # DEBUG 
    # generated_points = generated_points * 10
    # print(generated_points)
    # print('-'*10)

    # Extremely important : places everyone in the observation box
    # TODO : Maybe find a better name for the function
    for coord in generated_points:
        coord[0] = generate_periodic_boundaries(coord[0], boxsize) # x coordinate
        coord[1] = generate_periodic_boundaries(coord[1], boxsize) # y coordinate
        # print(coord[0], coord[1])
    


    # ADD ANGLE INITIALISATION, POLARITY, NOISE, ETC.
    # Angle initialisation for every seed point (initialisation is done in radians)
    generated_angles = np.random.random(len(generated_points))* 2.0 * np.pi
    # Generate the associated polarity vectors
    generated_polarities = generate_every_polarity_vector(generated_angles)

    return generated_points, generated_angles, generated_polarities

def lattice_initialization(point_number, boxsize, lattice_noise=0):
    """
    Create seed points (Voronoi center) on a controlled or "noisy" (with a bit of noise) lattice, and applies periodic boundary conditions.
    TO DO: initialize polarity, angles, momentum, a flag indicating at which stage we are currently in.
    
    Parameters
    -----
    point_number : int
        Number of seed point to create
    boxsize : float
        Length of the axis for observation box (assumed square for now)
    lattice_noise : float
        Intensity of the noise applied to the lattice seed points

    Returns
    -----
    generated_points : ndarray of ndarray
        Array of seed points on a lattice (of coordinates x,y)
    generated_angles : ndarray
        Array of randomly drawn angles in radians
    generated_polarities : ndarray of double
        Array of polarity vectors (cos theta_i, sin theta_i), initialized with random initial values 
    (in the future maybe): flag
    """
    # FIXME Step should be computed and not hard coded
    generated_points = [[x + np.random.random() * lattice_noise, y + np.random.random() * lattice_noise] for x in range(int(-boxsize/2), int(boxsize/2),1) for y in range(int(-boxsize/2), int(boxsize/2),1)]


    # Extremely important : places everyone in the observation box
    # TODO : Maybe find a better name for the function
    for coord in generated_points:
        coord[0] = generate_periodic_boundaries(coord[0], boxsize) # x coordinate
        coord[1] = generate_periodic_boundaries(coord[1], boxsize) # y coordinate
        # print(coord[0], coord[1])
    

    generated_points = np.array(generated_points)
    # ADD ANGLE INITIALISATION, POLARITY, NOISE, ETC.
    # Angle initialisation for every seed point (initialisation is done in radians)
    generated_angles = np.random.random(len(generated_points))* 2.0 * np.pi
    # Generate the associated polarity vectors
    generated_polarities = generate_every_polarity_vector(generated_angles)

    return generated_points, generated_angles, generated_polarities



def generate_periodic_boundaries(val, box):
    """
    Generates periodic boundaries by translating values to fit inside the observation box.
    
    Inspired by Hiraiwa Tetsuya. Modified because of performance issues. 
    Parameters:
        val: double, coordinate
        box: size of one side of the observation box
    -----
    Returns:
        translated_value: double, coordinate after translation
    
    Note: in CCM.C (program by T.H), this function is called at initialization and [after Runge-Kutta but before polarity time evolution]
    """

    ## Just like in a videogame, when you cross a side of the screen, you appear back on the other side

    # while(val > box/2.0):
    #     # val = val - box
    #     # The above option is correct but time consuming
    #     val = val % box
    # while( val <= -box/2.0):
    #     # val = val + box
    #     # The above option is correct but time consuming
    #     val = val % box

    while(val < -box/2.0 or val > box/2.0):
        if(val > box/2.0 and val < box):
            val = (box % val) * (-1.0)
        elif(val < -box/2.0 and val > -box):
            val = (-box % val) * (-1.0)
        elif(val >= box):
            val = val % box
        elif(val <= box):
            val = val % -box
    return val




In [27]:
%%cython -a
import numpy as np
def generate_periodic_voronoi_cells_2( double [:,:] points, double boxsize):
    """
    Copy the initialized points around the observation box of size "boxsize", over the 8 nearest regions
    C-like version (250+ times faster than the pure python)

    Parameters:
    points : numpy array of arrays
        (x,y) coordinates of the Voronoi centers
    boxsize : float
        Size of a side of the observation box (assumed square-shaped)
    -----
    Returns:
    extended_points : numpy array of arrays
        (x,y) coordinates of the Voronoi centers and their copy
    """

    #Define all the variables
    cdef Py_ssize_t index
    cdef Py_ssize_t len_points = len(points)
    DTYPE=np.float64
    # Empty point array to store the original points and three copies
    extended_points_np = np.zeros((len(points)*9,2), dtype=DTYPE)
    cdef double [:,:] extended_points = extended_points_np

    index = 0
    # Copy the original values
    for index in range(len_points):
        extended_points[index,0] = points[index,0]
        extended_points[index,1] = points[index,1]

        # Translate along the y-axis
    for index in range(len_points):
        extended_points[index+len_points,0] = points[index,0]
        extended_points[index+len_points,1] = points[index,1] + boxsize
        extended_points[index+len_points*2,0] = points[index,0]
        extended_points[index+len_points*2,1] = points[index,1] - boxsize

    # Translate along the x-axis
    for index in range(len_points):
        extended_points[index+len_points*3,0] = points[index,0] + boxsize
        extended_points[index+len_points*3,1] = points[index,1]
        extended_points[index+len_points*4,0] = points[index,0] - boxsize
        extended_points[index+len_points*4,1] = points[index,1]

    # Translate along the diagonal
    for index in range(len_points):
        extended_points[index+len_points*5,0] = points[index,0] + boxsize
        extended_points[index+len_points*5,1] = points[index,1] + boxsize

        extended_points[index+len_points*6,0] = points[index,0] - boxsize
        extended_points[index+len_points*6,1] = points[index,1] - boxsize

        extended_points[index+len_points*7,0] = points[index,0] + boxsize
        extended_points[index+len_points*7,1] = points[index,1] - boxsize

        extended_points[index+len_points*8,0] = points[index,0] - boxsize
        extended_points[index+len_points*8,1] = points[index,1] + boxsize

    return extended_points



In [28]:
### IMPORTS
import matplotlib.pyplot as plt
from scipy.spatial import Voronoi, voronoi_plot_2d
import numpy as np
from matplotlib.patches import Ellipse

boxsize = 100
#np.random.seed(29308)
np.random.seed(4876)
points = lattice_initialization(100,boxsize)
extended_points = generate_periodic_voronoi_cells_2(points,boxsize)

vor = Voronoi(extended_points)
#regions, vertices = voronoi_finite_polygons_2d(vor, boxsize)
#index, perimeters, areas = get_polygons_properties_2d(regions, vertices)
#delta_r = 0.001
#K_A = 1
#K_P = 1
#A_0 = 1
#P_0 = 1

TypeError: a bytes-like object is required, not 'tuple'

In [29]:
def adder(point, force, polarity):
    point += 1
    force += 2
    polarity += 3
    
    return point, force, polarity

In [37]:
points = np.arange(10)
forces = np.arange(10,20)
polarities = np.arange(20,30)

print("Points : \n{}".format(points))
print("Forces : \n{}".format(forces))
print("Polarities : \n{}".format(polarities))

Points : 
[0 1 2 3 4 5 6 7 8 9]
Forces : 
[10 11 12 13 14 15 16 17 18 19]
Polarities : 
[20 21 22 23 24 25 26 27 28 29]


In [32]:
points_next, forces_next, polarities_next = adder(points, forces, polarities)

In [40]:
print("Points_next : \n{}".format(points_next))
print("Forces_next : \n{}".format(forces_next))
print("Polarities_next : \n{}".format(polarities_next))

Points_next : 
[ 1  2  3  4  5  6  7  8  9 10]
Forces_next : 
[12 13 14 15 16 17 18 19 20 21]
Polarities_next : 
[23 24 25 26 27 28 29 30 31 32]


In [63]:
sp_list = [x for x in range(10)]
fo_list = [x for x in range(10)]
po_list = [x for x in range(10)]

sp_np = np.arange(10)
fo_np = np.arange(10)
po_np = np.arange(10)

def test_list(sp,fo,po):
    sp_next = []
    fo_next = []
    po_next = []
    for s, f, p  in zip(sp,fo,po):
        s_n, f_n, p_n = adder(s,f,p)
        sp_next.append(s_n)
        fo_next.append(f_n)
        po_next.append(p_n)
        
    sp_next = np.array(sp_next)
    fo_next = np.array(fo_next)
    po_next = np.array(po_next)
    
    return sp_next, fo_next, po_next


def test_np(sp,fo,po):
    sp_next = np.zeros(len(sp)) 
    fo_next = np.zeros(len(fo)) 
    po_next = np.zeros(len(po)) 
    
    for index in range(len(sp)):
        sp_next[index], fo_next[index], po_next[index] = adder(sp[index], fo[index], po[index])
    
    return sp_next, fo_next, po_next

def test_np_2(sp,fo,po):
    sp_next = np.zeros(len(sp)) 
    fo_next = np.zeros(len(fo)) 
    po_next = np.zeros(len(po)) 
    
    sp_next[:], fo_next[:], po_next[:] = adder(sp[:], fo[:], po[:])
    
    return sp_next, fo_next, po_next

In [64]:
test_list(sp_list,fo_list,po_list)

(array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10]),
 array([ 2,  3,  4,  5,  6,  7,  8,  9, 10, 11]),
 array([ 3,  4,  5,  6,  7,  8,  9, 10, 11, 12]))

In [70]:
print(sp_np)
print('-'*30)
test_np(sp_np, fo_np, po_np)
print(sp_np)
print('-'*30)

[ 4  5  6  7  8  9 10 11 12 13]
------------------------------
[ 4  5  6  7  8  9 10 11 12 13]
------------------------------


In [71]:
print(sp_np)
print('-'*30)
test_np_2(sp_np, fo_np, po_np)
print(sp_np)
print('-'*30)

[ 4  5  6  7  8  9 10 11 12 13]
------------------------------
[ 5  6  7  8  9 10 11 12 13 14]
------------------------------


In [66]:
a,b,c = test_np_2(sp_np, fo_np, po_np) 
d,e,f = test_np(sp_np, fo_np, po_np)

a == d
print('-'*10)
b == e
print('-'*10)
c == f

----------
----------


array([False, False, False, False, False, False, False, False, False,
       False])

In [55]:
%timeit test_list(sp_list,fo_list,po_list)

709 µs ± 6.52 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)


In [56]:
%timeit test_np(sp_np, fo_np, po_np)

1.37 ms ± 39.2 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
